<a href="https://colab.research.google.com/github/monikamanjunath25/ScamShield/blob/main/ScamShield_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q scikit-learn gradio

In [5]:
import re
import gradio as gr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Sample training dataset
# 1 = Scam, 0 = Legitimate

messages = [
    # Scam messages
    "Your bank account is blocked. Verify your OTP now",
    "Congratulations you won a lottery. Pay fee to claim prize",
    "Click this link to update your bank KYC immediately",
    "Share your UPI PIN to receive your cashback",
    "Your account will close today. Click this link",
    "Send your OTP to receive your refund",
    "You have won a free gift. Pay processing charges",
    "Install this app to unblock your bank account",
    "Your electricity connection will be disconnected. Pay now",
    "Approve this UPI collect request to receive money",
    "Your ATM card is blocked. Share your CVV",
    "Urgent payment required to avoid account suspension",
    "You have received a prize. Enter your bank details",
    "Download this remote access app for bank verification",
    "Click suspicious link to claim free rewards",
    "Send money first to receive your job offer",
    "Your parcel is held. Pay a small delivery fee here",
    "Share your password to verify your bank account",
    "Your KYC expired. Enter your details immediately",
    "Transfer money to unlock your account",

    # Legitimate messages
    "Your bank statement is ready for download",
    "Your salary has been credited to your account",
    "Your UPI payment of 250 rupees was successful",
    "Your electricity bill payment was successful",
    "Your account balance is available in the bank app",
    "Your debit card transaction was completed",
    "Your bank branch will remain closed on Sunday",
    "Your monthly statement is ready",
    "Never share your OTP or UPI PIN with anyone",
    "Your college fee payment was successful",
    "Your refund has been processed",
    "Your bank has received your complaint",
    "Your transaction receipt is available",
    "Your loan application status has been updated",
    "Your account received a deposit",
    "Your bank appointment is confirmed",
    "Your payment was declined. Contact your bank through its official app",
    "Your fixed deposit maturity date is approaching",
    "Your account service request is complete",
    "Your monthly bill is ready to view"
]

labels = [1] * 20 + [0] * 20

# Create and train machine learning model
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(messages, labels)

# Suspicious patterns
rules = {
    "OTP or PIN request": r"\b(otp|upi pin|atm pin|cvv|password)\b",
    "Urgent pressure": r"\b(urgent|immediately|today|blocked|suspended)\b",
    "Suspicious link": r"(https?://|bit\.ly|tinyurl|click this link)",
    "Prize or reward offer": r"\b(lottery|won a prize|free gift|cashback)\b",
    "Payment demand": r"\b(processing fee|pay now|transfer money|send money)\b",
    "Remote access request": r"\b(anydesk|remote access|screen sharing)\b"
}

def detect_scam(message):

    if not message.strip():
        return "Please enter a message.", "No analysis available."

    probability = model.predict_proba([message])[0][1]

    reasons = []

    for reason, pattern in rules.items():
        if re.search(pattern, message, re.IGNORECASE):
            reasons.append(reason)

    # Combined prototype risk score
    score = int(probability * 65 + min(len(reasons), 3) * 12)
    score = min(score, 100)

    if score >= 65:
        result = "HIGH RISK"
    elif score >= 35:
        result = "MEDIUM RISK"
    else:
        result = "LOW RISK"

    explanation = (
        f"Risk score: {score}/100\n"
        f"Result: {result}\n\n"
        f"ML model scam score: {probability * 100:.1f}%\n\n"
        "Warning signs:\n"
    )

    if reasons:
        explanation += "\n".join(
            "- " + reason for reason in reasons
        )
    else:
        explanation += "- No listed warning signs detected."

    explanation += (
        "\n\nNote: This is an experimental demo. "
        "A low score does not guarantee safety. "
        "Always verify payments independently."
    )

    return result, explanation


# Create web interface
demo = gr.Interface(
    fn=detect_scam,
    inputs=gr.Textbox(
        lines=5,
        label="Enter SMS / WhatsApp message",
        placeholder="Paste a message to check..."
    ),
    outputs=[
        gr.Textbox(label="Fraud Risk"),
        gr.Textbox(label="AI Analysis")
    ],
    title="ScamShield - AI Scam Detector",
    description="Check suspicious financial messages using AI."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://188420c69ad4cfa5be.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
